In [38]:
!pip install numpy
import numpy as np 
import random as rd


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [70]:
def simulation_complete_1player(rolls, Ns, verbose=False, heuristic=1):
    progress = {s: 0 for s in range(2, 13)}
    turn_idx = 0
    n_turns = 0

    HEURISTICS = {
        1: heuristic_random_stop4,
        2: rule_of_28,
        3: make_max_steps_stop_k(1),
        4: make_max_steps_stop_k(2),
        5: make_max_steps_stop_k(3),
        6: make_max_steps_stop_k(4),
        7: make_max_steps_stop_k(5),
        }

    if heuristic not in HEURISTICS:
        raise ValueError(f"Unknown heuristic: {heuristic}")

    heuristic_fn = HEURISTICS[heuristic]

    while nb_won_columns(progress, Ns) < 3:
        if turn_idx >= len(rolls):
            raise ValueError("Not enough turn-roll lists in rolls to finish the simulation.")

        temp_progress = {}
        is_turn_over = False
        turn_roll_count = 0
        turn_rolls = rolls[turn_idx]
        roll_idx = 0

        if verbose:
            print("\n=== New turn ===")
            print("Secured progress:", progress)

        while not is_turn_over:
            if roll_idx >= len(turn_rolls):
                raise ValueError(
                    f"Not enough dice rolls in rolls[{turn_idx}] to finish this turn."
                )

            roll = turn_rolls[roll_idx]
            roll_idx += 1
            turn_roll_count += 1

            pairings = get_pairings(roll)

            legal_actions = [
                action for action in pairings
                if action_is_legal(action, progress, temp_progress, Ns)
            ]

            if verbose:
                print("\nRoll:", roll)
                print("Pairings:", pairings)
                print("Open columns:", list(temp_progress.keys()))
                print("Legal actions:", legal_actions)

            if not legal_actions:
                temp_progress = {}
                is_turn_over = True

                if verbose:
                    print("Bust! Temporary progress lost.")
                continue

            action, stop = heuristic_fn(
                progress,
                temp_progress,
                legal_actions,
                Ns,
                turn_roll_count
            )

            temp_progress = apply_action(action, progress, temp_progress, Ns)

            if verbose:
                print("Chosen action:", action)
                print("Temporary progress:", temp_progress)

            if stop:
                progress = bank_progress(progress, temp_progress, Ns)
                is_turn_over = True

                if verbose:
                    print("Player stops.")
                    print("New secured progress:", progress)

        n_turns += 1
        turn_idx += 1

    return n_turns, progress

In [40]:
def action_is_legal(action, progress, temp_progress, Ns):
    new_columns = set()

    for s in action:
        current_pos = progress[s] + temp_progress.get(s, 0)

        if current_pos >= Ns[s]:
            return False

        if s not in temp_progress:
            new_columns.add(s)

    return len(temp_progress) + len(new_columns) <= 3



In [41]:
def apply_action(action, progress, temp_progress, Ns):
    """
    Returns a new temp_progress after applying the action.
    """
    new_temp = temp_progress.copy()

    for s in action:
        current_pos = progress[s] + new_temp.get(s, 0)

        if current_pos < Ns[s]:
            new_temp[s] = new_temp.get(s, 0) + 1

            # cap so we never go beyond the top
            if progress[s] + new_temp[s] > Ns[s]:
                new_temp[s] = Ns[s] - progress[s]

    return new_temp

In [42]:
def bank_progress(progress, temp_progress, Ns):
    new_progress = progress.copy()

    for s, inc in temp_progress.items():
        new_progress[s] = min(new_progress[s] + inc, Ns[s])

    return new_progress

In [43]:
def nb_won_columns(progress, Ns):
    return sum(progress[s] >= Ns[s] for s in range(2, 13))

In [44]:
def should_stop(progress, temp_progress, Ns, stop_threshold=3):
    total_temp = sum(temp_progress.values())
    return total_temp >= stop_threshold

In [45]:
def get_pairings(roll):
    d1, d2, d3, d4 = roll

    pairings = [
        tuple(sorted((d1 + d2, d3 + d4))),
        tuple(sorted((d1 + d3, d2 + d4))),
        tuple(sorted((d1 + d4, d2 + d3))),
    ]

    # remove duplicates while preserving order
    unique_pairings = []
    seen = set()
    for p in pairings:
        if p not in seen:
            seen.add(p)
            unique_pairings.append(p)

    return unique_pairings

In [53]:
Ns = {
    2: 3,
    3: 5,
    4: 7,
    5: 9,
    6: 11,
    7: 13,
    8: 11,
    9: 9,
    10: 7,
    11: 5,
    12: 3
}
N_turns_max = 1000
rng = np.random.default_rng(0)
rolls = [[tuple(rng.integers(1, 7, size=4)) for _ in range(200)] for _ in range(N_turns_max)]

In [47]:
def heuristic_random_stop4(progress, temp_progress, legal_actions, Ns, turn_roll_count):
    """
    heuristic with random pairing & stop after 4 turns 
    """
    # random choice of pairing
    action = rd.choice(legal_actions)

    # stops after 4 turns
    stop = (turn_roll_count >= 4)

    return action, stop

In [48]:
def rule_of_28(progress, temp_progress, legal_actions, Ns, turn_roll_count):
    action = choose_action_rule_of_28(progress, temp_progress, legal_actions, Ns)
    new_temp_progress = apply_action(action, progress, temp_progress, Ns) # should decide whether or not to stop after playing the next game
    stop = should_stop_rule_of_28(new_temp_progress)
    return action, stop

def choose_action_rule_of_28(progress, temp_progress, legal_actions, Ns):
    """
    legal_actions: list of actions like (5, 7) or (8, 8)
    progress[s]: secured progress in column s
    temp_progress[s]: temporary progress in current turn
    """

    def column_move_weight(s):
        # 6 - |7 - s| : favors middle columns
        return 6 - abs(7 - s)

    def move_value(action):
        value = 0

        # Count how many times each column is advanced by this action
        # e.g. (7,7) -> advances column 7 twice
        counts = {}
        for s in action:
            counts[s] = counts.get(s, 0) + 1

        for s, p_i in counts.items():
            # marker(i) = 1 if a new neutral marker is opened in column s
            opens_new_marker = int(s not in temp_progress)

            value += p_i * column_move_weight(s)
            value -= 6 * opens_new_marker

        return value

    return max(legal_actions, key=move_value)

def should_stop_rule_of_28(temp_progress):
    """
    Stop if progress value >= 28, as described in the paper.
    temp_progress[s] = number of spaces advanced this turn in column s
    """

    if not temp_progress:
        return False

    # base progress value: sum_i (s_i + 1)(|7 - i| + 1)
    value = 0
    open_columns = list(temp_progress.keys())

    for s, s_i in temp_progress.items():
        value += (s_i + 1) * (abs(7 - s) + 1)

    # difficulty adjustments only matter when 3 neutral markers are open
    if len(open_columns) == 3:
        all_odd = all(s % 2 == 1 for s in open_columns)
        all_even = all(s % 2 == 0 for s in open_columns)
        all_high = all(s >= 7 for s in open_columns)
        all_low = all(s <= 7 for s in open_columns)

        if all_odd:
            value += 2
        if all_even:
            value -= 2
        if all_high:
            value += 4
        if all_low:
            value += 4

    return value >= 28

In [67]:
def make_max_steps_stop_k(k):
    def heuristic(progress, temp_progress, legal_actions, Ns, turn_roll_count):
        def score(action):
            val = 0
            for s in action:
                val += 6 - abs(7 - s)   # favorise les colonnes centrales

                # bonus si la colonne est proche d'être finie
                current_pos = progress[s] + temp_progress.get(s, 0)
                remaining = Ns[s] - current_pos
                if remaining <= 2:
                    val += 3

                # petit malus si on ouvre une nouvelle colonne
                if s not in temp_progress:
                    val -= 1

            return val

        action = max(legal_actions, key=score)
        stop = (turn_roll_count >= k)
        return action, stop
    return heuristic

In [54]:
simulation_complete_1player(rolls, Ns, verbose=True, heuristic=1)


=== New turn ===
Secured progress: {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

Roll: (np.int64(6), np.int64(4), np.int64(4), np.int64(2))
Pairings: [(np.int64(6), np.int64(10)), (np.int64(8), np.int64(8))]
Open columns: []
Legal actions: [(np.int64(6), np.int64(10)), (np.int64(8), np.int64(8))]
Chosen action: (np.int64(8), np.int64(8))
Temporary progress: {np.int64(8): 2}

Roll: (np.int64(2), np.int64(1), np.int64(1), np.int64(1))
Pairings: [(np.int64(2), np.int64(3))]
Open columns: [np.int64(8)]
Legal actions: [(np.int64(2), np.int64(3))]
Chosen action: (np.int64(2), np.int64(3))
Temporary progress: {np.int64(8): 2, np.int64(2): 1, np.int64(3): 1}

Roll: (np.int64(2), np.int64(5), np.int64(4), np.int64(6))
Pairings: [(np.int64(7), np.int64(10)), (np.int64(6), np.int64(11)), (np.int64(8), np.int64(9))]
Open columns: [np.int64(8), np.int64(2), np.int64(3)]
Legal actions: []
Bust! Temporary progress lost.

=== New turn ===
Secured progress: {2: 0, 3: 0, 4: 0, 

(160, {2: 2, 3: 0, 4: 7, 5: 9, 6: 4, 7: 13, 8: 11, 9: 3, 10: 3, 11: 4, 12: 0})

In [56]:
simulation_complete_1player(rolls, Ns, verbose=True, heuristic=2)


=== New turn ===
Secured progress: {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

Roll: (np.int64(6), np.int64(4), np.int64(4), np.int64(2))
Pairings: [(np.int64(6), np.int64(10)), (np.int64(8), np.int64(8))]
Open columns: []
Legal actions: [(np.int64(6), np.int64(10)), (np.int64(8), np.int64(8))]
Chosen action: (np.int64(8), np.int64(8))
Temporary progress: {np.int64(8): 2}

Roll: (np.int64(2), np.int64(1), np.int64(1), np.int64(1))
Pairings: [(np.int64(2), np.int64(3))]
Open columns: [np.int64(8)]
Legal actions: [(np.int64(2), np.int64(3))]
Chosen action: (np.int64(2), np.int64(3))
Temporary progress: {np.int64(8): 2, np.int64(2): 1, np.int64(3): 1}
Player stops.
New secured progress: {2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 2, 9: 0, 10: 0, 11: 0, 12: 0}

=== New turn ===
Secured progress: {2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 2, 9: 0, 10: 0, 11: 0, 12: 0}

Roll: (np.int64(6), np.int64(2), np.int64(3), np.int64(6))
Pairings: [(np.int64(8), np.int64(9)), (np.

(58, {2: 1, 3: 2, 4: 7, 5: 9, 6: 3, 7: 12, 8: 9, 9: 5, 10: 7, 11: 0, 12: 3})

In [74]:
simulation_complete_1player(rolls, Ns, verbose=True, heuristic=3)


=== New turn ===
Secured progress: {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}

Roll: (np.int64(6), np.int64(4), np.int64(4), np.int64(2))
Pairings: [(np.int64(6), np.int64(10)), (np.int64(8), np.int64(8))]
Open columns: []
Legal actions: [(np.int64(6), np.int64(10)), (np.int64(8), np.int64(8))]
Chosen action: (np.int64(8), np.int64(8))
Temporary progress: {np.int64(8): 2}
Player stops.
New secured progress: {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 2, 9: 0, 10: 0, 11: 0, 12: 0}

=== New turn ===
Secured progress: {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 2, 9: 0, 10: 0, 11: 0, 12: 0}

Roll: (np.int64(6), np.int64(2), np.int64(3), np.int64(6))
Pairings: [(np.int64(8), np.int64(9)), (np.int64(5), np.int64(12))]
Open columns: []
Legal actions: [(np.int64(8), np.int64(9)), (np.int64(5), np.int64(12))]
Chosen action: (np.int64(8), np.int64(9))
Temporary progress: {np.int64(8): 1, np.int64(9): 1}
Player stops.
New secured progress: {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0,

(23, {2: 3, 3: 0, 4: 1, 5: 9, 6: 3, 7: 12, 8: 11, 9: 4, 10: 0, 11: 1, 12: 2})